# Chapter 8 실습: RNN → LSTM → RNN+Attention → BERT 감정 분류 비교

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/glasslego/2026-ml-study/blob/main/notebooks/ch08_rnn_lstm_attention_bert.ipynb)

**혁펜하임 Easy! 딥러닝 Chapter 8**

이 노트북에서는 시퀀스 모델의 발전 과정을 **수식 → 코드 → 실험**으로 따라갑니다.

| 모델 | 핵심 아이디어 | 한계 |
|------|-------------|------|
| RNN | 순환 구조로 시퀀스 처리 | 장기 의존성 문제 (기울기 소실) |
| LSTM | 게이트로 정보 흐름 제어 | 순차 처리, 먼 거리 여전히 약함 |
| RNN + Attention | 모든 hidden state에 가중합 | h 자체가 뭉개진 표현 |
| BERT (Transformer) | Self-Attention으로 모든 토큰이 직접 참조 | 계산 비용 $O(n^2)$ |

---
## 0. 환경 설정

In [ ]:
# Colab 환경에서만 실행
import sys
if 'google.colab' in sys.modules:
    !pip install transformers -q

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 디바이스: {device}")

---
## 1. 데이터셋 & 어휘 사전

### 1.1 토크나이징과 정수 인코딩

자연어를 모델에 넣으려면 **문자열 → 정수 시퀀스**로 변환해야 합니다.

$$
\text{"i love this movie"} \xrightarrow{\text{tokenize}} [\text{"i"}, \text{"love"}, \text{"this"}, \text{"movie"}] \xrightarrow{\text{w2i}} [15, 22, 38, 24]
$$

### 1.2 임베딩 (Embedding)

정수 인덱스를 **밀집 벡터(dense vector)**로 변환합니다. One-hot은 차원이 어휘 크기와 같아서 비효율적이고, 단어 간 유사도 정보가 없습니다.

$$
\mathbf{e}_t = \mathbf{E}[x_t] \in \mathbb{R}^{d_{\text{embed}}}
$$

- $\mathbf{E} \in \mathbb{R}^{|V| \times d_{\text{embed}}}$: 임베딩 행렬 (학습 가능한 파라미터)
- $x_t$: 시점 $t$의 토큰 인덱스
- $d_{\text{embed}}$: 임베딩 차원 (이 실습에서는 32)

### 1.3 패딩 (Padding)

배치 처리를 위해 모든 문장의 길이를 `MAX_LEN`으로 맞춥니다.

$$
[15, 22, 38, 24] \xrightarrow{\text{pad to 12}} [15, 22, 38, 24, \underbrace{0, 0, 0, 0, 0, 0, 0, 0}_{\text{PAD}}]
$$

In [ ]:
# 간단한 영어 감정 분류 (긍정=1, 부정=0)
DATA = [
    # 긍정 (1)
    ("i love this movie it is amazing", 1),
    ("great film wonderful story", 1),
    ("really enjoyed watching this", 1),
    ("excellent acting and direction", 1),
    ("this is the best movie ever", 1),
    ("beautiful and touching story", 1),
    ("highly recommend this film", 1),
    ("fantastic performance by the cast", 1),
    ("loved every moment of it", 1),
    ("a masterpiece of cinema", 1),
    # 부정 (0)
    ("terrible movie waste of time", 0),
    ("i hate this boring film", 0),
    ("worst movie i have ever seen", 0),
    ("bad acting and poor story", 0),
    ("this film is absolutely awful", 0),
    ("do not watch this movie", 0),
    ("completely disappointed by this", 0),
    ("horrible and painful to watch", 0),
    ("not worth your time at all", 0),
    ("dull and uninspiring throughout", 0),
]

TRAIN = DATA[:16]
TEST  = DATA[16:]

# 어휘 사전 구축
all_words = [w for sent, _ in DATA for w in sent.split()]
vocab = ['<PAD>', '<UNK>'] + sorted(set(all_words))
w2i = {w: i for i, w in enumerate(vocab)}
VOCAB_SIZE = len(vocab)
PAD_IDX = 0
MAX_LEN = 12

print(f"어휘 크기: {VOCAB_SIZE}")
print(f"예시 인코딩 'i love this': {[w2i.get(w, 1) for w in 'i love this'.split()]}")


def encode(sentence, max_len=MAX_LEN):
    """문장 → 정수 인덱스 시퀀스 (패딩 포함)"""
    ids = [w2i.get(w, 1) for w in sentence.split()]
    ids = ids[:max_len] + [PAD_IDX] * max(0, max_len - len(ids))
    return ids


class SentimentDataset(Dataset):
    def __init__(self, data):
        self.X = torch.tensor([encode(s) for s, _ in data], dtype=torch.long)
        self.y = torch.tensor([label for _, label in data], dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.y[i]


train_loader = DataLoader(SentimentDataset(TRAIN), batch_size=8, shuffle=True)
test_loader  = DataLoader(SentimentDataset(TEST),  batch_size=4)

---
## 2. Model 1: Vanilla RNN

### 2.1 RNN의 핵심 수식

RNN은 **이전 시점의 hidden state** $\mathbf{h}_{t-1}$과 **현재 입력** $\mathbf{x}_t$를 결합하여 **현재 hidden state** $\mathbf{h}_t$를 만듭니다.

$$
\boxed{\mathbf{h}_t = \tanh(\mathbf{W}_{xh}\,\mathbf{x}_t + \mathbf{W}_{hh}\,\mathbf{h}_{t-1} + \mathbf{b}_h)}
$$

- $\mathbf{x}_t \in \mathbb{R}^{d_{\text{embed}}}$: 시점 $t$의 입력 임베딩
- $\mathbf{h}_t \in \mathbb{R}^{d_{\text{hidden}}}$: 시점 $t$의 hidden state
- $\mathbf{W}_{xh} \in \mathbb{R}^{d_{\text{hidden}} \times d_{\text{embed}}}$: 입력→hidden 가중치
- $\mathbf{W}_{hh} \in \mathbb{R}^{d_{\text{hidden}} \times d_{\text{hidden}}}$: hidden→hidden 가중치 (순환 연결)
- $\mathbf{b}_h \in \mathbb{R}^{d_{\text{hidden}}}$: 편향

### 2.2 Many-to-One 구조

감정 분류는 시퀀스 전체를 하나의 레이블로 분류하므로, **마지막 시점**의 hidden state만 사용합니다.

$$
\hat{y} = \text{softmax}(\mathbf{W}_o\,\mathbf{h}_T + \mathbf{b}_o)
$$

### 2.3 RNN의 한계: 기울기 소실 (Vanishing Gradient)

역전파 시 기울기는 $\mathbf{W}_{hh}$를 반복적으로 곱합니다.

$$
\frac{\partial \mathbf{h}_T}{\partial \mathbf{h}_1} = \prod_{t=2}^{T} \frac{\partial \mathbf{h}_t}{\partial \mathbf{h}_{t-1}} = \prod_{t=2}^{T} \text{diag}(\tanh'(\cdot)) \cdot \mathbf{W}_{hh}
$$

- $|\tanh'(x)| \leq 1$ 이므로, $T$가 크면 기울기가 **기하급수적으로 작아짐**
- 결과: 시퀀스 앞부분의 정보가 학습에 거의 반영되지 않음 → **"멀수록 잊혀진다"**

```
시점:   x₁ ──→ x₂ ──→ x₃ ──→ ... ──→ x_T ──→ ŷ
         │      │      │              │
         h₁ ──→ h₂ ──→ h₃ ──→ ... ──→ h_T ──→ FC ──→ softmax
         ▲                              │
         └── 이 정보가 h_T까지 살아남기 어려움 ──┘
```

In [ ]:
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden_dim=64, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(
            embed_dim, hidden_dim,
            batch_first=True,
            nonlinearity='tanh'  # h_t = tanh(W_xh * x_t + W_hh * h_{t-1} + b)
        )
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        emb = self.embedding(x)          # (batch, seq, embed_dim)
        out, h_n = self.rnn(emb)         # h_n: (1, batch, hidden_dim)
        # Many-to-One: 마지막 hidden state만 사용
        return self.fc(h_n.squeeze(0))   # (batch, num_classes)

---
## 3. Model 2: LSTM (Long Short-Term Memory)

### 3.1 LSTM이 해결하려는 문제

RNN의 기울기 소실 문제를 **게이트(gate)** 메커니즘으로 해결합니다. 핵심은 **Cell State** $\mathbf{c}_t$라는 별도의 기억 경로를 만든 것입니다.

### 3.2 LSTM의 3개 게이트 + Cell State

#### (1) Forget Gate (망각 게이트): "무엇을 버릴까?"

$$
\boxed{\mathbf{f}_t = \sigma(\mathbf{W}_f [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_f)}
$$

- $\sigma$: 시그모이드 함수 → 출력이 $(0, 1)$ 범위
- $\mathbf{f}_t \approx 0$: 이전 기억을 **버림**
- $\mathbf{f}_t \approx 1$: 이전 기억을 **유지**

#### (2) Input Gate (입력 게이트): "무엇을 기억할까?"

$$
\boxed{\mathbf{i}_t = \sigma(\mathbf{W}_i [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_i)}
$$

$$
\tilde{\mathbf{c}}_t = \tanh(\mathbf{W}_c [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_c)
$$

- $\mathbf{i}_t$: 새 정보를 얼마나 반영할지 결정
- $\tilde{\mathbf{c}}_t$: 새로 추가할 후보 기억

#### (3) Cell State 업데이트: "기억 갱신"

$$
\boxed{\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{c}}_t}
$$

- $\odot$: 원소별 곱 (element-wise multiplication)
- 첫째 항: 이전 기억 중 **유지할 부분**
- 둘째 항: 새로 **추가할 기억**
- 이 덧셈 구조 덕분에 기울기가 직접 흐를 수 있음 → **기울기 소실 완화**

#### (4) Output Gate (출력 게이트): "무엇을 출력할까?"

$$
\boxed{\mathbf{o}_t = \sigma(\mathbf{W}_o [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_o)}
$$

$$
\boxed{\mathbf{h}_t = \mathbf{o}_t \odot \tanh(\mathbf{c}_t)}
$$

### 3.3 RNN vs LSTM 정보 흐름 비교

```
RNN:   h₁ ──tanh──→ h₂ ──tanh──→ h₃ ──→ ... ──→ h_T
       └── 곱셈 반복 → 기울기 소실 ──────────────────┘

LSTM:  c₁ ──(+)───→ c₂ ──(+)───→ c₃ ──→ ... ──→ c_T  (Cell State: 덧셈 경로)
        │    ↑         │    ↑        │
       h₁   f·c+i·c̃  h₂   f·c+i·c̃  h₃           h_T  (Hidden State)
       └── 덧셈 경로 → 기울기가 직접 흐름 ─────────┘
```

### 3.4 LSTM의 남은 한계

- Cell State도 결국 **순차적으로** 업데이트 → 아주 긴 시퀀스에서는 여전히 약함
- $t=1$의 정보가 $t=T$에 도달하려면 $T-1$번의 게이트를 통과해야 함
- 병렬 처리 불가 (시점 $t$의 계산이 $t-1$에 의존)

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden_dim=64, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        emb = self.embedding(x)               # (batch, seq, embed_dim)
        out, (h_n, c_n) = self.lstm(emb)      # h_n: hidden, c_n: cell state
        # Many-to-One: 마지막 hidden state 사용
        return self.fc(h_n.squeeze(0))         # (batch, num_classes)

---
## 4. Model 3: RNN + Attention

### 4.1 Attention의 동기

RNN/LSTM의 Many-to-One 구조는 **마지막 hidden state** $\mathbf{h}_T$ 하나에 모든 정보를 압축합니다. 이것은 **정보 병목(information bottleneck)**입니다.

$$
\text{RNN/LSTM: } \quad \underbrace{\mathbf{h}_T}_{\text{모든 정보를 이 하나에 압축}} \rightarrow \hat{y}
$$

Attention은 **모든 시점의 hidden state**를 활용하여 이 병목을 해소합니다.

### 4.2 Attention 메커니즘 수식

#### Step 1: Score 계산

각 시점 $t$의 hidden state $\mathbf{h}_t$에 대한 **중요도 점수(score)**를 계산합니다.

$$
e_t = \mathbf{w}_a^\top \mathbf{h}_t + b_a \quad \text{(학습 가능한 선형 변환)}
$$

- $\mathbf{w}_a \in \mathbb{R}^{d_{\text{hidden}}}$: attention 가중치 벡터
- 원래 Bahdanau Attention에서는 $e_t = \mathbf{v}^\top \tanh(\mathbf{W}_s \mathbf{s} + \mathbf{W}_h \mathbf{h}_t)$ (디코더 상태 $\mathbf{s}$와 비교)
- 분류 문제에서는 디코더가 없으므로, 학습 가능한 벡터 $\mathbf{w}_a$와의 내적으로 단순화

#### Step 2: Softmax로 가중치 정규화

$$
\boxed{\alpha_t = \frac{\exp(e_t)}{\sum_{k=1}^{T} \exp(e_k)}}
$$

- $\alpha_t \in (0, 1)$: 시점 $t$의 **attention 가중치**
- $\sum_{t=1}^{T} \alpha_t = 1$: 가중치의 합은 1

#### Step 3: Context Vector (가중합)

$$
\boxed{\mathbf{c} = \sum_{t=1}^{T} \alpha_t \cdot \mathbf{h}_t}
$$

- $\mathbf{c}$: context vector — 모든 시점의 정보를 **중요도에 따라 가중합**한 벡터
- 이 $\mathbf{c}$를 최종 분류에 사용: $\hat{y} = \text{softmax}(\mathbf{W}_o \mathbf{c} + \mathbf{b}_o)$

### 4.3 Attention의 의미

```
문장: "i love this movie it is amazing"

RNN (Many-to-One):
  h₁  h₂  h₃  h₄  h₅  h₆  h₇
                              ↓      ← h₇만 사용
                             FC → ŷ

RNN + Attention:
  h₁  h₂  h₃  h₄  h₅  h₆  h₇
  ↓   ↓   ↓   ↓   ↓   ↓   ↓
  α₁  α₂  α₃  α₄  α₅  α₆  α₇  ← 각 시점별 가중치
  └────────┬────────────────┘
           c = Σ αᵢhᵢ           ← 가중합
           ↓
          FC → ŷ
```

### 4.4 양방향(Bidirectional) LSTM 사용

단방향 RNN은 $\mathbf{h}_t$가 $x_1, \ldots, x_t$만 반영합니다. **양방향 LSTM**은 순방향/역방향을 결합합니다.

$$
\overrightarrow{\mathbf{h}}_t = \text{LSTM}_{\text{forward}}(\mathbf{x}_t, \overrightarrow{\mathbf{h}}_{t-1})
$$

$$
\overleftarrow{\mathbf{h}}_t = \text{LSTM}_{\text{backward}}(\mathbf{x}_t, \overleftarrow{\mathbf{h}}_{t+1})
$$

$$
\mathbf{h}_t = [\overrightarrow{\mathbf{h}}_t \| \overleftarrow{\mathbf{h}}_t] \in \mathbb{R}^{2 \cdot d_{\text{hidden}}}
$$

### 4.5 Attention의 한계

- $\mathbf{h}_t$ 자체가 RNN/LSTM이 만든 것 → 이미 **뭉개진(compressed) 표현**
- Attention은 뭉개진 $\mathbf{h}_t$들의 가중합일 뿐, 원본 토큰 간 직접 상호작용은 없음
- 이 한계를 극복한 것이 → **Self-Attention (Transformer)**

In [ ]:
class RNNAttentionClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden_dim=64, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        # 양방향 LSTM: 앞뒤 문맥 모두 활용
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        # Attention score 계산: e_t = w_a^T h_t + b_a
        self.attn = nn.Linear(hidden_dim * 2, 1)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        emb = self.embedding(x)                           # (batch, seq, embed)
        out, _ = self.lstm(emb)                           # (batch, seq, hidden*2)

        # Step 1: Score 계산
        attn_scores = self.attn(out)                      # (batch, seq, 1)
        # Step 2: Softmax → 가중치 정규화
        attn_weights = torch.softmax(attn_scores, dim=1)  # (batch, seq, 1)
        # Step 3: Context vector = 가중합
        context = (attn_weights * out).sum(dim=1)         # (batch, hidden*2)

        return self.fc(context)

    def get_attention(self, x):
        """시각화용: 각 단어의 attention 가중치 반환"""
        with torch.no_grad():
            emb = self.embedding(x)
            out, _ = self.lstm(emb)
            weights = torch.softmax(self.attn(out), dim=1)
        return weights.squeeze(-1)  # (batch, seq)

---
## 5. Model 4: BERT (Transformer Encoder)

### 5.1 RNN 계열의 근본적 한계

RNN + Attention도 결국:
1. $\mathbf{h}_t$는 RNN이 **순차적으로** 만든 뭉개진 벡터
2. 토큰 $x_i$와 $x_j$가 직접 상호작용하지 못함 (항상 $\mathbf{h}$를 경유)
3. 순차 처리 → 병렬화 불가

### 5.2 Self-Attention: 모든 토큰이 서로 직접 참조

Transformer는 각 토큰을 **Query, Key, Value** 세 벡터로 변환합니다.

$$
\mathbf{Q} = \mathbf{X}\mathbf{W}_Q, \quad \mathbf{K} = \mathbf{X}\mathbf{W}_K, \quad \mathbf{V} = \mathbf{X}\mathbf{W}_V
$$

- $\mathbf{X} \in \mathbb{R}^{T \times d_{\text{model}}}$: 입력 임베딩 행렬
- $\mathbf{W}_Q, \mathbf{W}_K \in \mathbb{R}^{d_{\text{model}} \times d_k}$: Query, Key 프로젝션
- $\mathbf{W}_V \in \mathbb{R}^{d_{\text{model}} \times d_v}$: Value 프로젝션

#### Scaled Dot-Product Attention

$$
\boxed{\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d_k}}\right)\mathbf{V}}
$$

- $\mathbf{Q}\mathbf{K}^\top \in \mathbb{R}^{T \times T}$: 모든 토큰 쌍의 유사도 행렬
- $\sqrt{d_k}$로 나누는 이유: 내적 값이 $d_k$에 비례하여 커지므로, softmax가 포화되는 것을 방지
- 결과: 각 토큰이 **다른 모든 토큰의 Value를 가중합**하여 새로운 표현을 얻음

#### 직관적 이해

```
문장: "i love this movie"

           i    love  this  movie
    i    [0.1   0.6   0.1   0.2 ]   ← "i"는 "love"에 가장 주목
    love [0.3   0.1   0.1   0.5 ]   ← "love"는 "movie"에 주목
    this [0.1   0.1   0.1   0.7 ]   ← "this"는 "movie"에 주목
    movie[0.1   0.5   0.2   0.2 ]   ← "movie"는 "love"에 주목
          └── QK^T를 softmax한 attention 행렬 ──┘
```

### 5.3 Multi-Head Attention

하나의 attention이 아닌, **여러 관점(head)**으로 동시에 바라봅니다.

$$
\text{head}_i = \text{Attention}(\mathbf{X}\mathbf{W}_Q^{(i)}, \, \mathbf{X}\mathbf{W}_K^{(i)}, \, \mathbf{X}\mathbf{W}_V^{(i)})
$$

$$
\boxed{\text{MultiHead}(\mathbf{X}) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) \, \mathbf{W}_O}
$$

- Head 1: 문법적 관계에 주목 (주어-동사)
- Head 2: 의미적 관계에 주목 (감정어-대상)
- Head 3: 인접 단어 관계에 주목
- ...

### 5.4 Transformer Encoder Block

하나의 인코더 블록은 다음과 같습니다:

$$
\mathbf{Z} = \text{LayerNorm}(\mathbf{X} + \text{MultiHead}(\mathbf{X}))
$$

$$
\mathbf{H} = \text{LayerNorm}(\mathbf{Z} + \text{FFN}(\mathbf{Z}))
$$

여기서 FFN(Feed-Forward Network)은:

$$
\text{FFN}(\mathbf{z}) = \text{GELU}(\mathbf{z}\mathbf{W}_1 + \mathbf{b}_1)\mathbf{W}_2 + \mathbf{b}_2
$$

- **Residual Connection** ($\mathbf{X} + \cdots$): 기울기가 직접 흐르는 지름길
- **Layer Normalization**: 학습 안정화
- BERT-base: 이 블록을 **12번** 쌓음

### 5.5 BERT의 입력: 위치 정보 추가

Self-Attention은 순서를 모름 (순환 구조가 없으므로). **Positional Encoding**으로 위치 정보를 추가합니다.

$$
\text{PE}(pos, 2i) = \sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)
$$

$$
\text{PE}(pos, 2i+1) = \cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)
$$

$$
\text{입력} = \text{Token Embedding} + \text{Positional Encoding} + \text{Segment Embedding}
$$

### 5.6 BERT vs RNN 계열 핵심 차이

| | RNN/LSTM | RNN+Attention | BERT (Transformer) |
|---|---------|--------------|--------------------|
| 토큰 간 거리 | $O(T)$ 시점 거쳐야 함 | $O(T)$ (h가 순차 생성) | $O(1)$ 직접 참조 |
| 병렬 처리 | 불가 (순차) | 불가 (순차) | 가능 (행렬 곱) |
| 표현력 | $\mathbf{h}_T$에 압축 | $\sum \alpha_i \mathbf{h}_i$ (뭉개진 h) | 각 토큰이 독립적으로 쇄신 |
| 사전학습 | 없음 | 없음 | 대규모 코퍼스로 사전학습 |
| 계산 비용 | $O(T)$ | $O(T)$ | $O(T^2)$ |

### 5.7 Encoder vs Decoder

Transformer 원 논문("Attention is All You Need")은 **Encoder-Decoder** 구조입니다.

```
┌─────────────────────────────────────────────────────────────────┐
│                    Transformer 전체 구조                         │
│                                                                 │
│   Encoder (왼쪽)              Decoder (오른쪽)                   │
│   ┌──────────────┐           ┌──────────────┐                   │
│   │ Self-Attention│           │ Masked       │                   │
│   │ (양방향)      │           │ Self-Attention│ ← 미래 토큰 차단 │
│   ├──────────────┤           ├──────────────┤                   │
│   │ FFN          │           │ Cross-Attn   │ ← Encoder 출력 참조│
│   └──────────────┘           ├──────────────┤                   │
│                              │ FFN          │                   │
│                              └──────────────┘                   │
│                                                                 │
│   입력: "I love this"         출력: "나는 이것을 사랑한다"        │
└─────────────────────────────────────────────────────────────────┘
```

#### Encoder (BERT가 사용)
- **양방향** Self-Attention: 모든 토큰이 앞뒤 모든 토큰을 참조
- 용도: 분류, NER, 문장 임베딩 등 **이해(understanding)** 태스크
- 대표 모델: **BERT**, RoBERTa, ELECTRA

#### Decoder (GPT가 사용)
- **Masked(Causal)** Self-Attention: 현재 위치 이전의 토큰만 참조
- 용도: 텍스트 생성, 번역 등 **생성(generation)** 태스크
- 대표 모델: **GPT** 시리즈, Claude

$$
\text{Masked Attention: } \text{softmax}\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d_k}} + \mathbf{M}\right)\mathbf{V}
$$

$$
\text{where } M_{ij} = \begin{cases} 0 & \text{if } i \geq j \\ -\infty & \text{if } i < j \end{cases}
$$

- 마스크 $\mathbf{M}$이 미래 위치를 $-\infty$로 만들어 softmax 후 0이 됨
- 이 실습의 BERT(DistilBERT)는 **Encoder만** 사용합니다

In [ ]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.optim import AdamW


class BertDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=32):
        self.data      = data
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        text, label = self.data[i]
        enc = self.tokenizer(
            text, max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'label':          torch.tensor(label)
        }

---
## 6. 학습 & 평가 함수

In [ ]:
def train_model(model, loader, epochs=60, lr=0.001, name=""):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    history = []

    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0, 0, 0
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(X)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            correct += (logits.argmax(1) == y).sum().item()
            total += len(y)

        acc = correct / total
        history.append(acc)
        if (epoch + 1) % 20 == 0:
            print(f"  [{name}] Epoch {epoch+1:3d} | Loss {total_loss:.4f} | Train Acc {acc:.0%}")

    return history


def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            preds = model(X).argmax(1)
            correct += (preds == y).sum().item()
            total += len(y)
    return correct / total

---
## 7. 학습 실행

### 7.1 RNN / LSTM / RNN+Attention 학습

In [ ]:
print("=" * 55)
print("Model 1: RNN  (h_t = tanh(W_xh·x_t + W_hh·h_{t-1} + b))")
print("=" * 55)
rnn_model = RNNClassifier(VOCAB_SIZE)
rnn_hist  = train_model(rnn_model, train_loader, epochs=60, name="RNN")
rnn_acc   = evaluate(rnn_model, test_loader)
print(f"  -> Test Accuracy: {rnn_acc:.0%}\n")

print("=" * 55)
print("Model 2: LSTM  (Forget/Input/Output Gate + Cell State)")
print("=" * 55)
lstm_model = LSTMClassifier(VOCAB_SIZE)
lstm_hist  = train_model(lstm_model, train_loader, epochs=60, name="LSTM")
lstm_acc   = evaluate(lstm_model, test_loader)
print(f"  -> Test Accuracy: {lstm_acc:.0%}\n")

print("=" * 55)
print("Model 3: RNN + Attention  (c = sum(alpha_i * h_i))")
print("=" * 55)
attn_model = RNNAttentionClassifier(VOCAB_SIZE)
attn_hist  = train_model(attn_model, train_loader, epochs=60, name="Attention")
attn_acc   = evaluate(attn_model, test_loader)
print(f"  -> Test Accuracy: {attn_acc:.0%}\n")

### 7.2 BERT (DistilBERT) 학습

DistilBERT는 BERT를 **Knowledge Distillation**으로 경량화한 모델입니다.
- BERT-base: 12 layers, 110M params
- DistilBERT: 6 layers, 66M params (40% 작지만 성능 97% 유지)

사전학습된 모델을 가져와서 **Fine-tuning** (마지막 분류 레이어만 학습)합니다.

In [ ]:
print("=" * 55)
print("Model 4: BERT (DistilBERT - Self-Attention 기반)")
print("=" * 55)
print("모델 다운로드 중...")

tokenizer  = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
bert_model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=2
).to(device)

bert_train_loader = DataLoader(BertDataset(TRAIN, tokenizer), batch_size=8, shuffle=True)
bert_test_loader  = DataLoader(BertDataset(TEST,  tokenizer), batch_size=4)

bert_optimizer = AdamW(bert_model.parameters(), lr=2e-5)
bert_hist = []

for epoch in range(5):  # BERT는 사전학습 덕분에 소량 데이터 + 적은 epoch으로 충분
    bert_model.train()
    total_loss, correct, total = 0, 0, 0
    for batch in bert_train_loader:
        ids   = batch['input_ids'].to(device)
        mask  = batch['attention_mask'].to(device)
        label = batch['label'].to(device)

        bert_optimizer.zero_grad()
        outputs = bert_model(input_ids=ids, attention_mask=mask, labels=label)
        outputs.loss.backward()
        bert_optimizer.step()

        preds = outputs.logits.argmax(1)
        total_loss += outputs.loss.item()
        correct += (preds == label).sum().item()
        total += len(label)

    acc = correct / total
    bert_hist.append(acc)
    print(f"  [BERT] Epoch {epoch+1} | Loss {total_loss:.4f} | Train Acc {acc:.0%}")

# BERT 테스트 정확도
bert_model.eval()
correct, total = 0, 0
with torch.no_grad():
    for batch in bert_test_loader:
        ids   = batch['input_ids'].to(device)
        mask  = batch['attention_mask'].to(device)
        label = batch['label'].to(device)
        preds = bert_model(input_ids=ids, attention_mask=mask).logits.argmax(1)
        correct += (preds == label).sum().item()
        total += len(label)
bert_acc = correct / total
print(f"  -> Test Accuracy: {bert_acc:.0%}\n")

---
## 8. 결과 비교

In [ ]:
print("\n" + "=" * 60)
print("최종 결과 비교 (테스트 정확도)")
print("=" * 60)
results = [
    ("RNN",           rnn_acc,  "h_T만 사용. 기울기 소실로 먼 정보 손실"),
    ("LSTM",          lstm_acc, "게이트+Cell State. 기울기 소실 완화"),
    ("RNN+Attention", attn_acc, "모든 h에 가중합. 정보 병목 완화"),
    ("BERT",          bert_acc, "Self-Attention. 토큰 간 직접 참조"),
]
for name, acc, desc in results:
    bar = '█' * int(acc * 20)
    print(f"  {name:<16} {acc:>4.0%} {bar:<20}  {desc}")

In [ ]:
# 학습 곡선 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# RNN/LSTM/Attention 학습 곡선 (60 epochs)
axes[0].plot(rnn_hist,  label='RNN', linestyle='--', alpha=0.7)
axes[0].plot(lstm_hist, label='LSTM', linestyle='-.', alpha=0.7)
axes[0].plot(attn_hist, label='RNN+Attention', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Train Accuracy')
axes[0].set_title('RNN / LSTM / RNN+Attention')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0.3, 1.05)

# BERT 학습 곡선 (5 epochs)
axes[1].plot(range(1, len(bert_hist)+1), bert_hist, 'o-', color='tab:red', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Train Accuracy')
axes[1].set_title('BERT (DistilBERT) - 5 epochs only')
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0.3, 1.05)

plt.tight_layout()
plt.show()

---
## 9. Attention 시각화

RNN+Attention 모델이 **어떤 단어에 주목**했는지 시각화합니다.

$$
\alpha_t = \frac{\exp(e_t)}{\sum_k \exp(e_k)} \quad \Rightarrow \quad \text{높은 } \alpha_t = \text{"이 단어가 분류에 중요"}
$$

In [ ]:
def visualize_attention(sentence, model=attn_model):
    """Attention 가중치를 텍스트 + 히트맵으로 시각화"""
    tokens = sentence.split()
    x = torch.tensor([encode(sentence)], dtype=torch.long).to(device)
    model.eval()
    weights = model.get_attention(x)[0, :len(tokens)].cpu().numpy()

    # 텍스트 출력
    print(f"\n문장: '{sentence}'")
    print("단어별 Attention 가중치:")
    for word, w in zip(tokens, weights):
        bar = '█' * max(1, int(w * 200))
        print(f"  {word:15s}  {w:.4f}  {bar}")

    # 히트맵
    fig, ax = plt.subplots(figsize=(max(6, len(tokens) * 0.8), 1.5))
    ax.imshow(weights.reshape(1, -1), cmap='YlOrRd', aspect='auto')
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(tokens, fontsize=12)
    ax.set_yticks([])
    for i, w in enumerate(weights):
        ax.text(i, 0, f'{w:.3f}', ha='center', va='center', fontsize=10)
    ax.set_title(f'Attention Weights: "{sentence}"')
    plt.tight_layout()
    plt.show()


visualize_attention("this is the best movie ever")
visualize_attention("terrible movie waste of time")
visualize_attention("i hate this boring film")

---
## 10. 새 문장 예측

4개 모델에 동일한 문장을 넣어 예측 결과를 비교합니다. 특히 **애매한 문장**에서 모델 간 차이가 드러납니다.

In [ ]:
def predict(model, sentence, model_type='rnn'):
    model.eval()
    if model_type == 'bert':
        enc = tokenizer(sentence, return_tensors='pt', max_length=32,
                        padding='max_length', truncation=True)
        with torch.no_grad():
            logits = model(
                input_ids=enc['input_ids'].to(device),
                attention_mask=enc['attention_mask'].to(device)
            ).logits
    else:
        x = torch.tensor([encode(sentence)], dtype=torch.long).to(device)
        with torch.no_grad():
            logits = model(x)

    prob = torch.softmax(logits, dim=1)
    pred = logits.argmax(1).item()
    label = "Positive" if pred == 1 else "Negative"
    conf  = prob[0][pred].item()
    return f"{label} ({conf:.0%})"


test_sentences = [
    "this movie is absolutely amazing",     # 명확한 긍정
    "i completely hate this awful film",    # 명확한 부정
    "not bad but not great either",         # 애매한 문장
    "the story was boring but acting good", # 혼합
]

print("\n" + "=" * 60)
print("새 문장 예측 비교")
print("=" * 60)
for s in test_sentences:
    print(f"\n입력: '{s}'")
    print(f"  RNN           -> {predict(rnn_model,  s)}")
    print(f"  LSTM          -> {predict(lstm_model, s)}")
    print(f"  RNN+Attention -> {predict(attn_model, s)}")
    print(f"  BERT          -> {predict(bert_model, s, 'bert')}")

---
## 11. 핵심 개념 정리

### 모델별 수식 요약

| 모델 | 핵심 수식 | 정보 접근 방식 |
|------|----------|---------------|
| **RNN** | $\mathbf{h}_t = \tanh(\mathbf{W}_{xh}\mathbf{x}_t + \mathbf{W}_{hh}\mathbf{h}_{t-1} + \mathbf{b})$ | $\mathbf{h}_T$만 사용 (순차 압축) |
| **LSTM** | $\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{c}}_t$ | Cell State로 장기 기억 유지 |
| **RNN+Attn** | $\mathbf{c} = \sum \alpha_t \mathbf{h}_t, \; \alpha_t = \text{softmax}(e_t)$ | 모든 시점 h에 가중합 |
| **BERT** | $\text{Attn}(\mathbf{Q},\mathbf{K},\mathbf{V}) = \text{softmax}(\frac{\mathbf{QK}^T}{\sqrt{d_k}})\mathbf{V}$ | 모든 토큰이 직접 참조 |

### 발전 과정에서 해결된 문제들

```
RNN     ──→  "기울기 소실로 장기 의존성 학습 불가"
  │
  ▼ Cell State 덧셈 경로 추가
LSTM    ──→  "기울기 소실 완화, but 순차 처리 + 먼 거리 여전히 약함"
  │
  ▼ 모든 hidden state에 가중합 (Attention)
RNN+Attn ──→  "정보 병목 완화, but h 자체가 뭉개진 표현"
  │
  ▼ RNN 제거 + Self-Attention으로 직접 참조
BERT    ──→  "O(1) 거리, 병렬 처리, 사전학습으로 소량 데이터 OK"
```

### Encoder vs Decoder 정리

| | Encoder (BERT) | Decoder (GPT) | Encoder-Decoder (T5, 번역) |
|---|---------------|--------------|---------------------------|
| Attention 방향 | 양방향 (전체 참조) | 단방향 (왼쪽만) | Encoder 양방향 + Decoder 단방향 + Cross |
| 주요 태스크 | 분류, NER, QA | 텍스트 생성 | 번역, 요약 |
| 이 실습 | DistilBERT ✓ | - | - |